# 06 · Model comparison and error analysis
Owners: **Member 3** (comparison) and **Member 4** (error analysis). Everyone reviews.

Run after notebooks 02–05 have written `results/<model>/`. No GPU needed: this only reads predictions.
All outputs go to `results/comparison/` and feed the *Results*, *Discussion* and *Error Analysis* sections.

## 0 · Setup
Run this cell first. On **Google Colab** it clones the repo, installs requirements and copies the
data from your Google Drive. Locally it does nothing special.

**Before the first run:** put `Train.csv`, `Test.csv` and `SampleSubmission.csv` (from Zindi) in a
Google Drive folder called `swahili_news`, and set `REPO_URL` below.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_URL = "https://github.com/YOUR-GROUP/swahili-news-sequential.git"   # <-- edit once
    REPO_DIR = Path("/content/swahili-news-sequential")
    DRIVE_DATA = Path("/content/drive/MyDrive/swahili_news")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    for name in ("Train.csv", "Test.csv", "SampleSubmission.csv"):
        shutil.copy(DRIVE_DATA / name, REPO_DIR / "data" / name)
    os.environ["CACHE_DIR"] = str(DRIVE_DATA / "cache")             # e.g. downloaded word vectors persist
    os.environ["OUTPUT_BACKUP_DIR"] = str(DRIVE_DATA / "outputs")   # results backed up to Drive
else:
    if Path.cwd().name == "notebooks":
        os.chdir(Path.cwd().parent)
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import itertools
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.metrics import confusion_matrix

from src import config as C
from src.data import get_splits, split_frames
from src.evaluate import compute_metrics, expected_calibration_error, mcnemar, show_log
from src.text import tokenize

sns.set_theme(style="whitegrid")
OUT = C.RESULTS_DIR / "comparison"
OUT.mkdir(exist_ok=True)
ORDER = ["baseline_tfidf_lr", "baseline_fasttext", "textcnn", "birnn_attention", "transformer"]
KIND = {"baseline_tfidf_lr": "Baseline: sparse n-grams", "baseline_fasttext": "Baseline: neural bag (order-free)",
        "textcnn": "Sequential: local (CNN)", "birnn_attention": "Sequential: recurrent + attention",
        "transformer": "Sequential: pretrained self-attention"}
keys = [k for k in ORDER if (C.RESULTS_DIR / k / "metrics.json").exists()]
print("Models found:", keys)
M = {k: json.loads((C.RESULTS_DIR / k / "metrics.json").read_text()) for k in keys}
P = {k: pd.read_csv(C.RESULTS_DIR / k / "predictions.csv") for k in keys}
names = {k: M[k]["model"] for k in keys}
df = get_splits()
_, _, te = split_frames(df)
te["n_words"] = te.content.map(lambda t: len(tokenize(t)))
for k in keys:                                   # every model must be scored on the same articles
    assert (P[k].id.values == te.id.values).all(), k
y = te.label.map({c: i for i, c in enumerate(C.CLASSES)}).values
probs = {k: P[k][[f"p_{c}" for c in C.CLASSES]].values for k in keys}

## 1 · Main comparison table (held-out test split, 95% bootstrap intervals)

In [ ]:
def ci(k, m):
    lo, hi = M[k].get(f"{m}_ci95", [np.nan, np.nan])
    return f"{M[k][m]:.3f} [{lo:.3f}, {hi:.3f}]"


table = pd.DataFrame({
    "model": [names[k] for k in keys], "approach": [KIND[k] for k in keys],
    "log loss ↓": [ci(k, "log_loss") for k in keys], "accuracy": [ci(k, "accuracy") for k in keys],
    "macro-F1": [ci(k, "macro_f1") for k in keys],
    "ROC-AUC (ovr)": [round(M[k]["roc_auc_ovr_macro"], 3) for k in keys],
    "ECE ↓": [round(M[k]["ece"], 3) for k in keys],
    "trainable params": [M[k].get("n_trainable_params", "-") for k in keys],
    "selected exp.": [M[k].get("selected_experiment", "-") for k in keys]})
table.to_csv(OUT / "comparison_table.csv", index=False)
(OUT / "comparison_table.md").write_text(table.to_markdown(index=False))
table

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for a, m in zip(ax, ["log_loss", "accuracy", "macro_f1"]):
    v = [M[k][m] for k in keys]
    err = np.array([[M[k][m] - M[k][f"{m}_ci95"][0], M[k][f"{m}_ci95"][1] - M[k][m]] for k in keys]).T
    a.barh([names[k] for k in keys], v, xerr=err, color=[f"C{i}" for i in range(len(keys))], capsize=3)
    a.set_title(m + (" (lower is better)" if m == "log_loss" else "")); a.invert_yaxis()
fig.tight_layout(); fig.savefig(OUT / "comparison_bars.png", dpi=150); plt.show()

## 2 · Experiment progression (validation split)

In [ ]:
log = show_log()
log.to_csv(OUT / "all_experiments.csv", index=False)
fig, ax = plt.subplots(figsize=(13, 4))
for i, (m, g) in enumerate(log.groupby("model", sort=False)):
    ax.plot(g.exp_id, g.val_log_loss, "o-", label=m, color=f"C{i}")
ax.set_ylabel("validation log loss"); ax.set_title("All experiments, per model (lower is better)")
ax.tick_params(axis="x", rotation=60); ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(OUT / "experiment_progression.png", dpi=150); plt.show()
log[["exp_id", "member", "model", "description", "val_log_loss", "val_accuracy", "val_macro_f1", "observation"]]

## 3 · Per-class F1: the minority classes decide macro-F1
Remember: the test split contains only 8 *kimataifa* and 3 *burudani* articles, so one article
changes those F1 values by a lot.

In [ ]:
f1 = pd.DataFrame({names[k]: pd.read_csv(C.RESULTS_DIR / k / "per_class.csv", index_col=0)["f1-score"] for k in keys})
f1.index = [f"{c} ({C.ENGLISH[c]}, n={int((y == i).sum())})" for i, c in enumerate(C.CLASSES)]
f1.to_csv(OUT / "per_class_f1.csv")
fig, ax = plt.subplots(figsize=(1.7 * len(keys) + 3, 3.8))
sns.heatmap(f1, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1, ax=ax)
ax.set_title("Per-class F1 (test)"); fig.tight_layout(); fig.savefig(OUT / "per_class_f1.png", dpi=150); plt.show()

## 4 · Confusions

In [ ]:
fig, axes = plt.subplots(1, len(keys), figsize=(4.2 * len(keys), 3.8))
axes = np.atleast_1d(axes)
pairs = []
for a, k in zip(axes, keys):
    cm = confusion_matrix(P[k].true, P[k].pred, labels=C.CLASSES)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=a, xticklabels=C.CLASSES, yticklabels=C.CLASSES)
    a.set_title(names[k], fontsize=9); a.tick_params(labelsize=7)
    cm_off = cm.copy()
    np.fill_diagonal(cm_off, 0)
    off = pd.DataFrame(cm_off, index=C.CLASSES, columns=C.CLASSES)
    for (t, p), n in off.stack().sort_values(ascending=False).head(4).items():
        if n:
            pairs.append({"model": names[k], "true": t, "predicted": p, "count": int(n)})
fig.tight_layout(); fig.savefig(OUT / "confusions_all_models.png", dpi=150); plt.show()
pairs = pd.DataFrame(pairs)
pairs.to_csv(OUT / "most_confused_pairs.csv", index=False)
pairs

> **Discussion prompt:** EDA section 7 predicted *kitaifa ↔ biashara* as the most similar pair
> (centroid cosine 0.91). Is it the dominant confusion for every model? Does sequence modelling reduce
> it compared with the bag-of-words baselines?

## 5 · Statistical significance (exact McNemar test on paired predictions)

In [ ]:
mc = pd.DataFrame(index=[names[k] for k in keys], columns=[names[k] for k in keys], dtype=float)
for a, b in itertools.combinations(keys, 2):
    r = mcnemar(y, probs[a].argmax(1), probs[b].argmax(1))
    mc.loc[names[a], names[b]] = mc.loc[names[b], names[a]] = r["p_value"]
mc.to_csv(OUT / "mcnemar_pvalues.csv")
mc.round(4)

## 6 · Calibration: why log loss and accuracy can rank models differently

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], "k--", lw=.8)
for i, k in enumerate(keys):
    conf, corr = probs[k].max(1), probs[k].argmax(1) == y
    bins = np.linspace(0, 1, 11)
    xs, ys = zip(*[(conf[m].mean(), corr[m].mean()) for lo, hi in zip(bins[:-1], bins[1:])
                   if (m := (conf > lo) & (conf <= hi)).sum() >= 5])
    ax.plot(xs, ys, "o-", ms=3, label=f"{names[k]} (ECE {expected_calibration_error(y, probs[k]):.3f})")
ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.legend(fontsize=7); ax.set_title("Reliability (test)")
fig.tight_layout(); fig.savefig(OUT / "reliability_all.png", dpi=150); plt.show()

## 7 · Does article length matter? (the sequential question)
If the sequence models handle long articles better than the baselines, or the transformer degrades
on articles past its 512-token limit, it shows up here.

In [ ]:
q = pd.qcut(te.n_words, 5)
acc_len = pd.DataFrame({names[k]: pd.Series(probs[k].argmax(1) == y).groupby(q.values, observed=True).mean()
                        for k in keys})
acc_len.index = [f"{iv.left:.0f}–{iv.right:.0f}" for iv in acc_len.index]
acc_len.to_csv(OUT / "accuracy_by_length.csv")
ax = acc_len.plot(marker="o", figsize=(8, 3.8))
ax.set_xlabel("article length (words), quintiles"); ax.set_ylabel("accuracy"); ax.set_title("Accuracy by article length")
plt.tight_layout(); plt.savefig(OUT / "accuracy_by_length.png", dpi=150); plt.show()
acc_len.round(3)

## 8 · Hard articles: wrong for all or most models
Read them and categorise each one: *genuinely mixed topic* (e.g. a sports-business sponsorship
story), *label noise*, *very short / empty*, *truncation (decisive information beyond the window)*, or
*other*. Fill the `category` column of the saved CSV: this manual analysis is the core of the
error-analysis section.

In [ ]:
wrong = np.stack([probs[k].argmax(1) != y for k in keys], 1)
te["n_models_wrong"] = wrong.sum(1)
for k in keys:
    te[f"pred_{k}"] = [C.CLASSES[i] for i in probs[k].argmax(1)]
hard = te.sort_values(["n_models_wrong", "n_words"], ascending=[False, True]).head(15)
hard_out = hard[["id", "label", "n_models_wrong", "n_words"] + [f"pred_{k}" for k in keys]].copy()
hard_out["snippet"] = hard.content.str.strip().str[:300]
hard_out["category"] = ""
hard_out.to_csv(OUT / "hard_articles_to_categorise.csv", index=False)
for _, r in hard.head(6).iterrows():
    print(f"\n[{r.id}] true = {r.label} ({C.ENGLISH[r.label]}) | wrong for {r.n_models_wrong}/{len(keys)} | "
          f"{r.n_words} words | predictions: {[r[f'pred_{k}'] for k in keys]}")
    print("   ", r.content.strip()[:400], "...")

## 9 · Near-duplicates and label noise (from EDA section 9)
Articles in the test split that have a near-duplicate in training: are they easier (leakage)? Are
conflicting-label duplicates among the errors?

In [ ]:
nd = pd.read_csv(C.RESULTS_DIR / "eda" / "near_duplicates.csv")
te["near_dup"] = te.id.isin(nd.id)
best_key = min(keys, key=lambda k: M[k]["log_loss"])
cmp = te.groupby("near_dup").apply(lambda g: (probs[best_key][g.index].argmax(1) == y[g.index]).mean())
print(f"Best model ({names[best_key]}) accuracy on test articles with / without a near-duplicate:")
print(cmp.rename({True: "has near-duplicate", False: "no near-duplicate"}).round(3).to_string(),
      f"\n(n = {int(te.near_dup.sum())} articles with a near-duplicate)")

## 10 · Confidently wrong predictions of the best model
Log loss punishes these most: a single wrong prediction at 99% confidence costs as much as dozens of
correct ones. Are they label noise, mixed-topic articles, or real model errors?

In [ ]:
pb = probs[best_key]
cw = te.assign(pred=[C.CLASSES[i] for i in pb.argmax(1)], confidence=pb.max(1), p_true=pb[np.arange(len(y)), y])
cw = cw[cw.pred != cw.label].sort_values("confidence", ascending=False).head(10)
cw["loss_contribution"] = -np.log(np.clip(cw.p_true, 1e-7, 1))
cw["snippet"] = cw.content.str.strip().str[:200]
cw[["id", "label", "pred", "confidence", "p_true", "loss_contribution", "near_dup", "snippet"]] \
    .to_csv(OUT / "confidently_wrong_best_model.csv", index=False)
share = cw.loss_contribution.sum() / (-np.log(np.clip(pb[np.arange(len(y)), y], 1e-7, 1))).sum()
print(f"The 10 most confident errors account for {share:.1%} of {names[best_key]}'s total test log loss")
cw[["id", "label", "pred", "confidence", "loss_contribution", "snippet"]]

## 11 · Cost vs performance

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for i, k in enumerate(keys):
    n = M[k].get("n_trainable_params")
    if isinstance(n, (int, float)) and n:
        ax.scatter(n, M[k]["log_loss"], s=60, color=f"C{i}", label=names[k])
ax.set_xscale("log"); ax.set_xlabel("trainable parameters (log)"); ax.set_ylabel("test log loss")
ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(OUT / "params_vs_logloss.png", dpi=150); plt.show()

## 12 · (Exploratory) Complementary errors: probability-averaging ensemble
**Not** used for model selection; it only motivates future work.

In [ ]:
seq_keys = [k for k in ["textcnn", "birnn_attention", "transformer"] if k in keys]
if len(seq_keys) >= 2:
    ens = compute_metrics(y, np.mean([probs[k] for k in seq_keys], 0), len(C.CLASSES))
    (OUT / "ensemble_exploratory.json").write_text(json.dumps(ens, indent=2))
    print("Mean of", seq_keys, "->", {k: round(v, 4) for k, v in ens.items()})

In [ ]:
from src.utils import persist_outputs
persist_outputs()